In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests
import json

import psycopg2 as pg
from sqlalchemy import create_engine

In [2]:
# Function to extract search results from Google within a date range
def get_search_results(query, num_pages=1, start_date=None, end_date=None):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    search_results = []
    for page in range(num_pages):
        date_range = ''
        if start_date and end_date:
            date_range = f'&tbs=cdr:1,cd_min:{start_date},cd_max:{end_date}'
        
        url = f"https://www.google.com/search?q={query}&start={page*10}{date_range}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        results = soup.find_all('div', class_='g')
        for result in results:
            link = result.find('a', href=True)
            if link:
                title = result.find('h3')
                title_text = title.text if title else 'No title'
                search_results.append({
                    'title': title_text,
                    'link': link['href'],
                    'date_accessed': datetime.now().strftime('%Y-%m-%d')
                })
    return search_results

# Function to extract content and published date from an article
def get_article_content_and_date(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extract content
    paragraphs = soup.find_all('p')
    content = ' '.join([para.text for para in paragraphs])
    
    # Attempt to find the published date
    published_date = 'Unknown'
    # Check common meta tag for published date
    date_meta = soup.find('meta', {'name': 'pubdate'}) or \
                soup.find('meta', {'property': 'article:published_time'}) or \
                soup.find('meta', {'name': 'date'})
    if date_meta:
        published_date = date_meta.get('content', 'Unknown')
    else:
        # Look for date in article text, if possible
        date_text = soup.find_all(string=lambda string: string and "published" in string.lower())
        if date_text:
            published_date = date_text[0].strip()
    
    return content, published_date

# Main function to get speeches using multiple keywords
def get_bri_speeches(keywords=["direktur Bank Rakyat Indonesia (BRI) berbicara"], num_pages=1, start_date=None, end_date=None):
    speeches = []
    for keyword in keywords:
        search_results = get_search_results(keyword, num_pages, start_date, end_date)
        for result in search_results:
            try:
                content, published_date = get_article_content_and_date(result['link'])
                speeches.append({
                    'title': result['title'],
                    'link': result['link'],
                    'date_accessed': result['date_accessed'],
                    'date_published': published_date,
                    'content': content
                })
            except Exception as e:
                print(f"Failed to extract content from {result['link']}: {e}")
    return speeches

In [3]:
# Define date range for the search
start_date = (datetime.now() - timedelta(days=3650)).strftime('%m/%d/%Y')
end_date = datetime.now().strftime('%m/%d/%Y')

keywords = ["direktur utama BRI", "dirut BRI", "Bank Rakyat Indonesia"]
speeches = get_bri_speeches(keywords, num_pages=20, start_date=start_date, end_date=end_date)
speeches_df = pd.DataFrame(speeches)
speeches_df.shape[0]
speeches_df.head(3)

Failed to extract content from https://mediacenter.riau.go.id/read/81046/seleksi-dirut-bri-syariah-resmi-dibuka-beriku.html: HTTPSConnectionPool(host='mediacenter.riau.go.id', port=443): Max retries exceeded with url: /read/81046/seleksi-dirut-bri-syariah-resmi-dibuka-beriku.html (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
Failed to extract content from https://bririns.or.id/about/management: HTTPSConnectionPool(host='bririns.or.id', port=443): Max retries exceeded with url: /about/management (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000192D217A200>, 'Connection to bririns.or.id timed out. (connect timeout=None)'))
Failed to extract content from https://www.uny.ac.id/index.php/id/berita/penandatanganan-nota-kesepahaman-dengan-bank-rakyat-indonesia: HTTPSConnectionPool(host='www.uny.ac.id', port=443): Max retries exceeded with u

,title,link,date_accessed,date_published,content
0,Dirut BRI Sunarso Dinobatkan The Best CEO in D...,https://bri.co.id/in/web/ppid/detail-news?titl...,2024-08-22,Unknown,Carilah jawaban pada halaman FAQ atau arahkan ...
1,Dirut BRI Sunarso Dinobatkan The Best CEO in D...,https://bri.co.id/in/web/ppid/detail-news?titl...,2024-08-22,Unknown,Carilah jawaban pada halaman FAQ atau arahkan ...
2,RUPST BRI: Susunan Direksi Tetap dan Hanya Sat...,https://infobanknews.com/rupst-bri-susunan-dir...,2024-08-22,2024-03-01T10:31:50+00:00,Jakarta – PT Bank Rakyat Indonesia (Persero) T...


In [4]:
speeches_df.shape

(496, 5)

In [5]:
pattern = r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\+\d{2}:\d{2}$'

# Filter the DataFrame
filtered_df = speeches_df[speeches_df['date_published'].str.match(pattern, na=False)]

In [6]:
filtered_df

,title,link,date_accessed,date_published,content
2,RUPST BRI: Susunan Direksi Tetap dan Hanya Sat...,https://infobanknews.com/rupst-bri-susunan-dir...,2024-08-22,2024-03-01T10:31:50+00:00,Jakarta – PT Bank Rakyat Indonesia (Persero) T...
4,BRI tidak rombak jajaran direksi dalam RUPST 2024,https://www.antaranews.com/berita/3991251/bri-...,2024-08-22,2024-03-01T20:03:23+07:00,Pewarta: Bayu SaputraEditor: Faisal Yunianto\t...
6,"BRI (BBRI) Rombak Pengurus, Ini Susunan Komisa...",https://finansial.bisnis.com/read/20240301/90/...,2024-08-22,2024-03-01T17:45:02+07:00,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...
7,BRI Umumkan Susunan Direksi Baru di RUPST 2024,https://biz.kompas.com/read/2024/03/02/0909163...,2024-08-22,2024-03-02T02:09:16+00:00,BRI Umumkan Susunan Direksi Baru di RUPST 2024...
9,"Gelar RUPST, Sunarso Kembali Ditunjuk Jadi Dir...",https://ekonomi.republika.co.id/berita/rrgb3o4...,2024-08-22,2023-03-13T15:47:48+07:00,RUPST sepakat mengangkat kembali direksi yang ...
...,...,...,...,...,...
449,Proses Seleksi dan Rekrutmen pada PT. Bank Rak...,https://suarausu.or.id/proses-seleksi-dan-rekr...,2024-08-22,2023-10-27T13:31:09+00:00,Sumber: brandingkan.com Penulis: Nazlina Dwi P...
455,Dirut BRI Ungkap Rahasia di Balik Laba 2023 Te...,https://finansial.bisnis.com/read/20240131/90/...,2024-08-22,2024-01-31T10:04:27+07:00,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...
467,Dirut BRI Ungkap Rahasia di Balik Laba 2023 Te...,https://finansial.bisnis.com/read/20240131/90/...,2024-08-22,2024-01-31T10:04:27+07:00,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...
473,PT Bank Rakyat Indonesia (Persero) Tbk,https://goletskerja.com/pt-bank-rakyat-indones...,2024-08-22,2024-08-21T09:07:00+00:00,PT Bank Rakyat Indonesia (Persero) Tbk – Banya...


In [7]:
filtered_df.loc[:,'date_published'] = pd.to_datetime(filtered_df['date_published'], utc=True).dt.tz_convert('Asia/Jakarta').dt.date

In [8]:
# Rename the columns using the rename() method
filtered_df = filtered_df.rename(columns={
    'title': 'article_title',
    'link': 'article_link',
    'date_accessed': 'article_date_accessed',
    'date_published': 'article_date_published',
    'content': 'article_content'
})

In [11]:
filtered_df = filtered_df.drop_duplicates(subset=['article_title', 'article_content']).reset_index(drop=True)
filtered_df

,article_title,article_link,article_date_accessed,article_date_published,article_content
0,RUPST BRI: Susunan Direksi Tetap dan Hanya Sat...,https://infobanknews.com/rupst-bri-susunan-dir...,2024-08-22,2024-03-01,Jakarta – PT Bank Rakyat Indonesia (Persero) T...
1,BRI tidak rombak jajaran direksi dalam RUPST 2024,https://www.antaranews.com/berita/3991251/bri-...,2024-08-22,2024-03-01,Pewarta: Bayu SaputraEditor: Faisal Yunianto\t...
2,"BRI (BBRI) Rombak Pengurus, Ini Susunan Komisa...",https://finansial.bisnis.com/read/20240301/90/...,2024-08-22,2024-03-01,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...
3,BRI Umumkan Susunan Direksi Baru di RUPST 2024,https://biz.kompas.com/read/2024/03/02/0909163...,2024-08-22,2024-03-02,BRI Umumkan Susunan Direksi Baru di RUPST 2024...
4,"Gelar RUPST, Sunarso Kembali Ditunjuk Jadi Dir...",https://ekonomi.republika.co.id/berita/rrgb3o4...,2024-08-22,2023-03-13,RUPST sepakat mengangkat kembali direksi yang ...
...,...,...,...,...,...
75,Rekrutmen BRILiaN Banking Officer Program (BBO...,https://lokerbumn.com/penerimaan-brilian-banki...,2024-08-22,2024-08-20,PT Bank Rakyat Indonesia (Persero) Tbk adalah ...
76,Proses Seleksi dan Rekrutmen pada PT. Bank Rak...,https://suarausu.or.id/proses-seleksi-dan-rekr...,2024-08-22,2023-10-27,Sumber: brandingkan.com Penulis: Nazlina Dwi P...
77,Dirut BRI Ungkap Rahasia di Balik Laba 2023 Te...,https://finansial.bisnis.com/read/20240131/90/...,2024-08-22,2024-01-31,Nyaman tanpa iklan. Langganan BisnisPro Nyaman...
78,PT Bank Rakyat Indonesia (Persero) Tbk,https://goletskerja.com/pt-bank-rakyat-indones...,2024-08-22,2024-08-21,PT Bank Rakyat Indonesia (Persero) Tbk – Banya...


In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

# PostgreSQL connection details (set in .env, see .env.example)
db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

# Create the connection string
connection_string = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"


In [13]:
query = """ 
DO $$ 
BEGIN
  IF EXISTS (
    SELECT 1 
    FROM public.textual_data 
    WHERE article_title = %(article_title)s AND article_date_published = %(article_date_published)s
  ) THEN
    UPDATE public.textual_data
    SET
        article_title = %(article_title)s,
        article_link = %(article_link)s,
        article_date_accessed = %(article_date_accessed)s,
        article_date_published = %(article_date_published)s,
        article_content = %(article_content)s
    WHERE article_title = %(article_title)s AND article_date_published = %(article_date_published)s;
  ELSE
    INSERT INTO public.textual_data (
        article_title,
        article_link,
        article_date_accessed,
        article_date_published,
        article_content
     )
    VALUES (
        %(article_title)s,
        %(article_link)s,
        %(article_date_accessed)s,
        %(article_date_published)s,
        %(article_content)s
    );
  END IF;
END $$;
"""

cur = update_engine.cursor()
column_mapping = {column: column for column in list(filtered_df.columns)}

for i in range(len(filtered_df)):
  row_dict = {sql_col: int(filtered_df[df_col].iloc[i]) if isinstance(filtered_df[df_col].iloc[i], np.integer) else filtered_df[df_col].iloc[i] for df_col, sql_col in column_mapping.items()}
  cur.execute(query, row_dict)
update_engine.commit()

In [14]:
filtered_df.to_csv('articles_w_published_dates_new_22_aug.csv', index=False)